# Agent 第 19 课：Multi-Agent on Bedrock（Supervisor / Collaborator）

Phase 1 第 8 课你手写了 Manager / Worker 模式 —— 一个 agent 分派任务给几个子 agent。Bedrock 把这件事做成了一等公民：**Agent Collaboration**（2024 底 GA）。

学完这节你能回答：
1. 为什么要多 agent（单 agent 做不了什么）
2. Supervisor / Collaborator 在 Bedrock 里是什么 resource
3. 两种协作模式：`SUPERVISOR` vs `SUPERVISOR_ROUTER`
4. 什么时候别用多 agent

## 1. 为什么要多 agent

**单 agent 会在这些地方吃瘪**：

- **工具/KB 多到 prompt 塞不下**：一个 agent 挂 30 个工具，instruction + schema 能吃掉几 k token，还没开始推理就慢一半
- **角色冲突**：一个 agent 同时要"友好客服"和"严谨数据分析师"，两种风格互相干扰
- **领域差异大**：HR / 财务 / 技术 支持三个域，想让同一个 agent 都做好 = 都做不好
- **权限隔离**：不同子 agent 挂不同 IAM role，接触不同数据，便于合规

**单 agent 够用的信号**：工具 ≤ 10 个，只面向一个明确场景，不需要分租户隔离。硬拆反而是过度设计。

## 2. Bedrock 的协作模型

一个 **supervisor agent** + 若干 **collaborator agent**：

```
User ─► Supervisor Agent
         ├── instruction (当调度员，不要自己答)
         ├── agentCollaboration = SUPERVISOR | SUPERVISOR_ROUTER
         └── Collaborators:
              ├── hr_agent     (alias → HR 子 agent)
              ├── finance_agent (alias → 财务子 agent)
              └── tech_agent   (alias → 技术支持子 agent)
```

**每个 collaborator 本身就是一个普通 Bedrock Agent**（第 13-15 课那种），可以有自己的 action group / KB / guardrail。你只是通过 **alias** 把它"接"到 supervisor 上。

supervisor 的 orchestration LLM 会根据用户输入自动挑 collaborator、把子问题分派过去、拿回结果、再综合回答。**不需要你写路由代码**。

## 3. 两种模式：SUPERVISOR vs SUPERVISOR_ROUTER

| | SUPERVISOR | SUPERVISOR_ROUTER |
|---|---|---|
| supervisor 自己的角色 | 主 reasoner：**可以**自己综合多个 collaborator 的结果 | 纯路由器：**只**分发给一个 collaborator，不做综合 |
| 典型场景 | 跨域问题（"把上月开销发给 HR 做预算汇报") | 问题明显落在单一域（"帮我订会议室" → 直接路给 workplace agent） |
| 延迟 | 高（要跑 supervisor + 子 agent 两轮） | 低（几乎等于子 agent） |
| 成本 | 高 | 低 |

**默认从 SUPERVISOR_ROUTER 起步**。真出现"必须跨 collaborator 综合"才升级到 SUPERVISOR。

有个中间量：**`SUPERVISOR`** + 你在 instruction 里明确写"单一问题就直接 route，不要自己答" —— 实操里常用。

In [ ]:
import boto3, os, time, uuid, json
os.environ.setdefault("AWS_REGION", "us-west-2")

agent   = boto3.client("bedrock-agent")
runtime = boto3.client("bedrock-agent-runtime")
sts     = boto3.client("sts")
ACCOUNT = sts.get_caller_identity()["Account"]

MODEL = "anthropic.claude-sonnet-4-6-20260101-v1:0"
AGENT_ROLE = f"arn:aws:iam::{ACCOUNT}:role/AmazonBedrockExecutionRoleForAgents_Demo"

## 4. 最小 demo：一个 supervisor + 两个 collaborator

下面创建两个功能非常窄的子 agent（用 instruction 区分，没真工具，好跑通流程），再创建一个 supervisor 把它俩接起来。

In [ ]:
def create_child(name, instruction):
    r = agent.create_agent(
        agentName=f"{name}-{uuid.uuid4().hex[:4]}",
        agentResourceRoleArn=AGENT_ROLE,
        foundationModel=MODEL,
        instruction=instruction,
        idleSessionTTLInSeconds=600,
    )
    aid = r["agent"]["agentId"]
    agent.prepare_agent(agentId=aid)
    while agent.get_agent(agentId=aid)["agent"]["agentStatus"] != "PREPARED":
        time.sleep(2)
    alias = agent.create_agent_alias(agentId=aid, agentAliasName="live")
    # 等 alias ready
    time.sleep(3)
    return aid, alias["agentAlias"]["agentAliasArn"]

HR_ID, HR_ALIAS_ARN = create_child(
    "hr-bot",
    "You are an HR assistant. Only answer questions about HR policy (PTO, benefits, onboarding). "
    "Refuse anything else politely.",
)
TECH_ID, TECH_ALIAS_ARN = create_child(
    "tech-bot",
    "You are a tech support assistant. Only answer questions about IT support (VPN, laptop, SSO). "
    "Refuse anything else politely.",
)
print("HR   alias:", HR_ALIAS_ARN)
print("TECH alias:", TECH_ALIAS_ARN)

In [ ]:
# 创建 supervisor —— 注意 agentCollaboration 字段
sup = agent.create_agent(
    agentName=f"supervisor-{uuid.uuid4().hex[:4]}",
    agentResourceRoleArn=AGENT_ROLE,
    foundationModel=MODEL,
    instruction=(
        "You are a dispatcher. Do not answer questions yourself. "
        "Route HR topics (PTO, benefits) to the hr_agent, "
        "IT topics (VPN, laptop, SSO) to the tech_agent. "
        "If unclear, ask a single short clarifying question."
    ),
    agentCollaboration="SUPERVISOR_ROUTER",
    idleSessionTTLInSeconds=600,
)
SUP_ID = sup["agent"]["agentId"]
print("supervisor id =", SUP_ID)

In [ ]:
# 把两个 collaborator 挂上 supervisor
agent.associate_agent_collaborator(
    agentId=SUP_ID, agentVersion="DRAFT",
    agentDescriptor={"aliasArn": HR_ALIAS_ARN},
    collaboratorName="hr_agent",
    collaborationInstruction="Send HR-related questions here (PTO, benefits, onboarding).",
    relayConversationHistory="TO_COLLABORATOR",   # 把 user 对话转给 collaborator
)
agent.associate_agent_collaborator(
    agentId=SUP_ID, agentVersion="DRAFT",
    agentDescriptor={"aliasArn": TECH_ALIAS_ARN},
    collaboratorName="tech_agent",
    collaborationInstruction="Send IT support questions here (VPN, laptop, SSO).",
    relayConversationHistory="TO_COLLABORATOR",
)

# Prepare + alias
agent.prepare_agent(agentId=SUP_ID)
while agent.get_agent(agentId=SUP_ID)["agent"]["agentStatus"] != "PREPARED":
    time.sleep(2)
SUP_ALIAS = agent.create_agent_alias(agentId=SUP_ID, agentAliasName="dev")
SUP_ALIAS_ID = SUP_ALIAS["agentAlias"]["agentAliasId"]
print("supervisor ready, aliasId =", SUP_ALIAS_ID)

## 5. 调用 + 看 trace

调 supervisor 的 API 和第 13 课一样。看 trace 时你会看到多一类事件：`multiAgentCollaboratorInvocationInput/Output` —— 这就是 supervisor 发给 collaborator 的请求和回的结果。

In [ ]:
def ask_supervisor(text):
    sid = str(uuid.uuid4())
    stream = runtime.invoke_agent(
        agentId=SUP_ID, agentAliasId=SUP_ALIAS_ID,
        sessionId=sid, inputText=text,
        enableTrace=True,
    )
    answer = []
    for ev in stream["completion"]:
        if "chunk" in ev:
            answer.append(ev["chunk"]["bytes"].decode())
        elif "trace" in ev:
            t = ev["trace"]["trace"]
            if "orchestration" in t:
                step = t["orchestration"]
                if "invocationInput" in step:
                    inv = step["invocationInput"]
                    if "agentCollaboratorInvocationInput" in inv:
                        c = inv["agentCollaboratorInvocationInput"]
                        print(f"  [ROUTE -> {c.get('agentCollaboratorName')}] {c.get('input',{}).get('text','')[:120]}")
    return "".join(answer)

# print(ask_supervisor("我想知道我还剩多少 PTO 假期"))      # 应该路给 hr_agent
# print(ask_supervisor("我的 VPN 连不上了"))                # 应该路给 tech_agent
# print(ask_supervisor("今天北京天气怎么样？"))             # 两个都不沾，supervisor 应澄清或拒答

## 6. 关键参数细节

- **`relayConversationHistory`**：`TO_COLLABORATOR` 把 supervisor session 里的历史转给子 agent；`DISABLED` 不转（省 token，但 collaborator 看不到上下文）
- **collaborator 必须用 alias（不是 agent id）关联**：发版时只要把 alias 指向新版 collaborator，supervisor 不用动
- **collaborator 自己也能是 supervisor**：理论上无限嵌套。实操别超过 2 层，不然 debug 地狱
- **session 是 supervisor 的**：子 agent 的对话会被 AWS 自动聚合到这个 session 下
- **每个 collaborator 独立计费**：supervisor 一次调用 = supervisor 的 token + 被触发的 collaborator 的 token，成本叠加

## 7. 什么时候别用 multi-agent

反面意见要听：

- **延迟敏感**（客服、IDE 补全）：supervisor + collaborator 轻易 10s+ → 用户等不了
- **工具少**（< 10）：一个 agent 搞得定，别拆
- **问题总是落在同一域**：SUPERVISOR_ROUTER 白白多一跳
- **团队还没搞定单 agent 的 eval**：多 agent 故障点 ×N，没 eval 就是盲拆盲改

**一个务实的分叉点**：当 supervisor instruction 里"根据话题路由"已经变成了一个复杂 if/else，才升级到 multi-agent。否则一个带多 tool 的 single agent 往往更快更便宜。

## 8. 清理

In [ ]:
# 顺序：先 disassociate collaborators，再删 alias，再删 agent
for name in ("hr_agent", "tech_agent"):
    try:
        agent.disassociate_agent_collaborator(
            agentId=SUP_ID, agentVersion="DRAFT", collaboratorId=name
        )
    except Exception as e:
        print("disassoc skipped:", name, e)

for aid in (SUP_ID, HR_ID, TECH_ID):
    try:
        for a in agent.list_agent_aliases(agentId=aid).get("agentAliasSummaries", []):
            agent.delete_agent_alias(agentId=aid, agentAliasId=a["agentAliasId"])
    except Exception as e:
        print("alias cleanup:", e)
    try:
        agent.delete_agent(agentId=aid, skipResourceInUseCheck=True)
    except Exception as e:
        print("agent delete:", e)
print("done")

## 9. 工程直觉

1. **Collaborator 是 alias 接入**，supervisor 永远指向 alias。回滚 / 发版只动 alias，别动 supervisor 配置。
2. **SUPERVISOR_ROUTER 是默认值**。只有明确需要"跨域综合回答"才升 SUPERVISOR —— 每次综合多付一笔 supervisor LLM 的账。
3. **最多两层嵌套**。三层以上 trace 看不清、成本也难估。真的有大体系，不如拆成独立服务、用消息队列串。
4. **每个 collaborator 独立 IAM role**：合规上的天然好处 —— HR agent 的 Lambda 只能读 HR 库，技术 agent 只能读工单库，互不越权。
5. **eval 也要分层**：给每个 collaborator 建独立 eval set（能力），再给 supervisor 建路由 eval（这句话是否路给了正确的 collaborator）。混一起测就难定位。
6. **延迟翻倍的心理预期**。supervisor + collaborator 双 LLM 调用，用户感受明显慢。值得的时候值得，不值就别折腾。

---

下一课（Phase 2 收官）：**Cost & Performance 调优 + 生产上线 checklist** —— 把 11-19 讲的所有工件串成一张"能上线吗"的表。